In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
if not os.path.exists('/content/processed'):
    !tar -xzf "/content/drive/MyDrive/Applied-DL-2/Data/processed_upload.tar.gz" -C /content/
    print("Done extracting.")
else:
    print("Already extracted, skipping.")

Done extracting.


In [ ]:
!ls /content/processed

ls: cannot access '/content/processed': No such file or directory


In [ ]:
!ls /content/aac_32/

dev  train


In [ ]:
!git clone https://github.com/roh1thbharathi/applied-dl-project-2.git
%cd /content/applied-dl-project-2
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, '/content/applied-dl-project-2/src')
print("Done.")

Cloning into 'applied-dl-project-2'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 58 (delta 15), reused 50 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 37.32 KiB | 1.55 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/applied-dl-project-2
Done.


In [ ]:
import os

BASE = "/content/drive/MyDrive/CodecRobust"
os.makedirs(BASE, exist_ok=True)
print("Folder created at:", BASE)

Folder created at: /content/drive/MyDrive/CodecRobust


In [ ]:
!python src/smoke_test.py

  Smoke Test — Codec-Robust Deepfake Detector

[1/4] Imports...
  OK  — 10 codec classes

[2/4] Codec classes:
  [ 0] uncompressed None  kbps
  [ 1] mp3          32    kbps
  [ 2] mp3          64    kbps
  [ 3] mp3          128   kbps
  [ 4] aac          32    kbps
  [ 5] aac          64    kbps
  [ 6] aac          128   kbps
  [ 7] opus         16    kbps
  [ 8] opus         32    kbps
  [ 9] opus         64    kbps

[3/4] Forward pass...
  deepfake_logits : (4, 2)
  codec_logits    : (4, 10)
  embedding       : (4, 128)

[4/4] Losses & metrics...
  L_deepfake    = 0.6508
  L_codec       = 2.3803
  L_contrastive = 1.8417
  EER (random)  = 51.4%  (expect ~50%)
  AUC (random)  = 0.479  (expect ~0.5)

  ALL CHECKS PASSED
  Params  : 501,412
  Device  : cuda


In [ ]:
from google.colab import files
uploaded = files.upload()  # select all 3 txt files at once
print("Uploaded:", list(uploaded.keys()))

Saving ASVspoof2019.LA.cm.train.trn.txt to ASVspoof2019.LA.cm.train.trn.txt
Saving ASVspoof2019.LA.cm.eval.trl.txt to ASVspoof2019.LA.cm.eval.trl.txt
Saving ASVspoof2019.LA.cm.dev.trl.txt to ASVspoof2019.LA.cm.dev.trl.txt
Uploaded: ['ASVspoof2019.LA.cm.train.trn.txt', 'ASVspoof2019.LA.cm.eval.trl.txt', 'ASVspoof2019.LA.cm.dev.trl.txt']


In [ ]:
import os
import shutil

os.makedirs('/content/asvspoof/LA/ASVspoof2019_LA_cm_protocols', exist_ok=True)

for fname in ['ASVspoof2019.LA.cm.train.trn.txt',
              'ASVspoof2019.LA.cm.dev.trl.txt',
              'ASVspoof2019.LA.cm.eval.trl.txt']:
    shutil.move(f'/content/applied-dl-project-2/{fname}',
                f'/content/asvspoof/LA/ASVspoof2019_LA_cm_protocols/{fname}')

print("Done.")
!ls /content/asvspoof/LA/ASVspoof2019_LA_cm_protocols/

Done.
ASVspoof2019.LA.cm.dev.trl.txt	 ASVspoof2019.LA.cm.train.trn.txt
ASVspoof2019.LA.cm.eval.trl.txt


In [ ]:
ASV_ROOT       = "/content/asvspoof"
PROCESSED_ROOT = "/content"
RESULTS_DIR    = "/content/drive/MyDrive/CodecRobust/results/grl_only"

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Config ready.")
print("ASV_ROOT:", ASV_ROOT)
print("PROCESSED_ROOT:", PROCESSED_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)

Config ready.
ASV_ROOT: /content/asvspoof
PROCESSED_ROOT: /content
RESULTS_DIR: /content/drive/MyDrive/CodecRobust/results/grl_only


In [ ]:
!ls /content/uncompressed/train/ | head -10

LA_T_1000406.pt
LA_T_1004644.pt
LA_T_1007571.pt
LA_T_1007663.pt
LA_T_1011221.pt
LA_T_1013597.pt
LA_T_1021790.pt
LA_T_1023546.pt
LA_T_1029184.pt
LA_T_1031233.pt


In [ ]:
!ls /content/uncompressed/train/ | grep "LA_T_5796029"

In [ ]:
!ls /content/uncompressed/train/ | wc -l

3000


In [ ]:
import os

available = set(f.replace('.pt', '') for f in os.listdir('/content/uncompressed/train/'))
print(f"Available train files: {len(available)}")
print("Sample:", list(available)[:3])

Available train files: 3000
Sample: ['LA_T_1735385', 'LA_T_7187958', 'LA_T_6015645']


In [ ]:
import pandas as pd
from pathlib import Path

# Load protocol
proto_path = "/content/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
df = pd.read_csv(proto_path, sep=" ", header=None,
                 names=["speaker", "filename", "dash", "system_id", "label"])

# Filter to only available files
available = set(f.replace('.pt', '') for f in os.listdir('/content/uncompressed/train/'))
df_filtered = df[df['filename'].isin(available)].reset_index(drop=True)

print(f"Protocol total: {len(df)}")
print(f"After filtering to available: {len(df_filtered)}")

Protocol total: 25380
After filtering to available: 3000


In [ ]:
# Patch ASVspoof2019Preprocessed to filter to available files
import data_utils

original_init = data_utils.ASVspoof2019Preprocessed.__init__

def patched_init(self, asv_root, processed_root, split="train", contrastive=False):
    original_init(self, asv_root, processed_root, split, contrastive)
    # Filter records to only files that exist in preprocessed folder
    available = set(
        f.replace('.pt', '')
        for f in os.listdir(Path(processed_root) / 'uncompressed' / split)
    )
    before = len(self.records)
    self.records = self.records[self.records['filename'].isin(available)].reset_index(drop=True)
    print(f"  [Filter] {split}: {before} → {len(self.records)} (matched preprocessed files)")

data_utils.ASVspoof2019Preprocessed.__init__ = patched_init
print("Patch applied.")

Patch applied.


In [ ]:
from importlib import reload
import data_utils
loaders = get_dataloaders(
    asv_root=ASV_ROOT,
    preprocessed_root=PROCESSED_ROOT,
    batch_size=4,
    num_workers=0,
    contrastive=False,
    max_samples=100,
)

batch = next(iter(loaders['train']))
print("Waveform shape:", batch['waveform'].shape)
print("Labels:", batch['label'])
print("Codec indices:", batch['codec_idx'])
print("Dataloader OK.")

  [DataLoader] Using preprocessed data from: /content
  [Filter] train: 25380 → 3000 (matched preprocessed files)
  [DataLoader] train: 3,000 samples | 10 codec variants available
  [Filter] dev: 24844 → 3000 (matched preprocessed files)
  [DataLoader] dev  : 3,000 samples | 10 codec variants available
  [DataLoader] WARN: could not load eval: No preprocessed folders found under /content. Run: python src/preprocess_codecs.py --asv_root <root>
  [DataLoader] train capped at 100 samples
  [RAM] Preloading 100 clips × 10 codecs into RAM...
  [RAM] Done. HDD reads = 0 from now on.
  [DataLoader] dev capped at 100 samples
  [RAM] Preloading 100 clips × 10 codecs into RAM...
  [RAM] Done. HDD reads = 0 from now on.
Waveform shape: torch.Size([4, 32000])
Labels: tensor([0, 0, 0, 0])
Codec indices: tensor([9, 6, 8, 8])
Dataloader OK.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import time
from pathlib import Path

from model import CodecRobustDetector, ContrastiveLoss
from data_utils import get_dataloaders, N_CODEC_CLASSES
from evaluate import compute_eer, compute_auc

# ── Config ────────────────────────────────────────────────
ALPHA           = 0.5
BETA            = 0.0
USE_CONTRASTIVE = False
EPOCHS          = 30
BATCH_SIZE      = 32
LR              = 3e-4
MAX_SAMPLES     = 3000
EMBED_DIM       = 256
SAVE_EVERY      = 1   # save every epoch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Data ──────────────────────────────────────────────────
loaders = get_dataloaders(
    asv_root=ASV_ROOT,
    preprocessed_root=PROCESSED_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=0,
    contrastive=USE_CONTRASTIVE,
    max_samples=MAX_SAMPLES,
)

# ── Model ─────────────────────────────────────────────────
model = CodecRobustDetector(embed_dim=EMBED_DIM, n_codec_classes=N_CODEC_CLASSES).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR/20)
df_crit    = nn.CrossEntropyLoss()
codec_crit = nn.CrossEntropyLoss()
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

# ── Resume from latest checkpoint if exists ───────────────
out_dir    = Path(RESULTS_DIR)
start_epoch = 1
best_eer    = 1.0

latest_ckpt = sorted(out_dir.glob("checkpoint_epoch_*.pt"))
if latest_ckpt:
    ckpt = torch.load(latest_ckpt[-1], map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_eer    = ckpt['best_eer']
    print(f"Resumed from epoch {ckpt['epoch']}, best EER so far: {best_eer*100:.2f}%")
else:
    print("No checkpoint found, starting fresh.")

# ── GRL lambda schedule ───────────────────────────────────
def grl_lambda(step, total_steps, lam_max=1.0):
    p = step / max(total_steps, 1)
    return lam_max * (2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0)

# ── Training loop ─────────────────────────────────────────
total_steps = EPOCHS * len(loaders["train"])
step = (start_epoch - 1) * len(loaders["train"])

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0 = time.time()
    for batch in loaders["train"]:
        lam = grl_lambda(step, total_steps)
        model.set_lambda(lam)
        wf        = batch["waveform"].to(device)
        labels    = batch["label"].to(device)
        codec_idx = batch["codec_idx"].to(device)
        out       = model(wf)
        loss      = df_crit(out["deepfake_logits"], labels) + ALPHA * codec_crit(out["codec_logits"], codec_idx)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        step += 1
    scheduler.step()

    # ── Dev eval ──────────────────────────────────────────
    model.eval()
    scores, labs = [], []
    with torch.no_grad():
        for batch in loaders["dev"]:
            s = torch.softmax(model(batch["waveform"].to(device))["deepfake_logits"], -1)[:, 1]
            scores.append(s.cpu())
            labs.append(batch["label"])
    scores = torch.cat(scores).numpy()
    labs   = torch.cat(labs).numpy()
    eer    = compute_eer(labs, scores)
    auc    = compute_auc(labs, scores)

    print(f"Epoch {epoch:02d}/{EPOCHS} | EER={eer*100:.2f}% AUC={auc:.4f} | time={time.time()-t0:.1f}s")

    # ── Save best ─────────────────────────────────────────
    if eer < best_eer:
        best_eer = eer
        torch.save(model.state_dict(), out_dir / "best_model.pt")
        print(f"  ✓ Best EER: {best_eer*100:.2f}% — saved")

    # ── Save checkpoint every epoch ───────────────────────
    if epoch % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'eer': eer,
            'best_eer': best_eer,
        }, out_dir / f"checkpoint_epoch_{epoch:02d}.pt")
        print(f"  ✓ Checkpoint saved: epoch {epoch}")

print(f"\nDone. Best EER: {best_eer*100:.2f}%")

Device: cuda
  [DataLoader] Using preprocessed data from: /content
  [Filter] train: 25380 → 3000 (matched preprocessed files)
  [DataLoader] train: 3,000 samples | 10 codec variants available
  [Filter] dev: 24844 → 3000 (matched preprocessed files)
  [DataLoader] dev  : 3,000 samples | 10 codec variants available
  [DataLoader] WARN: could not load eval: No preprocessed folders found under /content. Run: python src/preprocess_codecs.py --asv_root <root>
Params: 550,948
No checkpoint found, starting fresh.
Epoch 01/30 | EER=22.57% AUC=0.8469 | time=142.3s
  ✓ Best EER: 22.57% — saved
  ✓ Checkpoint saved: epoch 1
Epoch 02/30 | EER=19.43% AUC=0.8653 | time=149.0s
  ✓ Best EER: 19.43% — saved
  ✓ Checkpoint saved: epoch 2
Epoch 03/30 | EER=12.83% AUC=0.9464 | time=149.5s
  ✓ Best EER: 12.83% — saved
  ✓ Checkpoint saved: epoch 3
Epoch 04/30 | EER=7.41% AUC=0.9784 | time=149.6s
  ✓ Best EER: 7.41% — saved
  ✓ Checkpoint saved: epoch 4
Epoch 05/30 | EER=8.15% AUC=0.9692 | time=149.4s
  ✓ 

In [38]:
!ls /content/drive/MyDrive/CodecRobust/results/grl_only/

best_model.pt		checkpoint_epoch_11.pt	checkpoint_epoch_22.pt
checkpoint_epoch_01.pt	checkpoint_epoch_12.pt	checkpoint_epoch_23.pt
checkpoint_epoch_02.pt	checkpoint_epoch_13.pt	checkpoint_epoch_24.pt
checkpoint_epoch_03.pt	checkpoint_epoch_14.pt	checkpoint_epoch_25.pt
checkpoint_epoch_04.pt	checkpoint_epoch_15.pt	checkpoint_epoch_26.pt
checkpoint_epoch_05.pt	checkpoint_epoch_16.pt	checkpoint_epoch_27.pt
checkpoint_epoch_06.pt	checkpoint_epoch_17.pt	checkpoint_epoch_28.pt
checkpoint_epoch_07.pt	checkpoint_epoch_18.pt	checkpoint_epoch_29.pt
checkpoint_epoch_08.pt	checkpoint_epoch_19.pt	checkpoint_epoch_30.pt
checkpoint_epoch_09.pt	checkpoint_epoch_20.pt
checkpoint_epoch_10.pt	checkpoint_epoch_21.pt


In [41]:
from model import CodecRobustDetector
from data_utils import N_CODEC_CLASSES

def load_model(checkpoint_path, device):
    model = CodecRobustDetector(embed_dim=256, n_codec_classes=N_CODEC_CLASSES).to(device)
    state = torch.load(checkpoint_path, map_location=device)
    if isinstance(state, dict) and 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'])
    else:
        model.load_state_dict(state)
    model.eval()
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

baseline  = load_model("/content/drive/MyDrive/CodecRobust/results/baseline/best_model.pt", device)
full      = load_model("/content/drive/MyDrive/CodecRobust/results/full_model/best_model.pt", device)
grl_only  = load_model("/content/drive/MyDrive/CodecRobust/results/grl_only/best_model.pt", device)

print("All 3 models loaded.")

All 3 models loaded.


In [46]:
!ls /content/uncompressed/dev/ | head -3

LA_D_1006568.pt
LA_D_1008730.pt
LA_D_1010295.pt


In [47]:
test_pt = torch.load('/content/uncompressed/dev/LA_D_1006568.pt', weights_only=True)
test_input = test_pt.unsqueeze(0).to(device)

with torch.no_grad():
    out = baseline(test_input)
    print("Logits:", out['deepfake_logits'])
    print("Softmax:", torch.softmax(out['deepfake_logits'], -1))

Logits: tensor([[-4.7022,  4.0766]], device='cuda:0')
Softmax: tensor([[1.5393e-04, 9.9985e-01]], device='cuda:0')


In [48]:
with torch.no_grad():
    out_grl = grl_only(test_input)
    print("GRL Logits:", out_grl['deepfake_logits'])
    print("GRL Softmax:", torch.softmax(out_grl['deepfake_logits'], -1))

with torch.no_grad():
    out_full = full(test_input)
    print("Full Logits:", out_full['deepfake_logits'])
    print("Full Softmax:", torch.softmax(out_full['deepfake_logits'], -1))

GRL Logits: tensor([[ 1.1574, -1.5076]], device='cuda:0')
GRL Softmax: tensor([[0.9349, 0.0651]], device='cuda:0')
Full Logits: tensor([[-4.0222,  4.0112]], device='cuda:0')
Full Softmax: tensor([[3.2435e-04, 9.9968e-01]], device='cuda:0')


In [49]:
import pandas as pd
from pathlib import Path

proto_path = Path(ASV_ROOT) / 'LA' / 'ASVspoof2019_LA_cm_protocols' / 'ASVspoof2019.LA.cm.dev.trl.txt'
df = pd.read_csv(proto_path, sep=' ', header=None,
                 names=['speaker','filename','dash','system_id','label'])

available = set(f.replace('.pt','') for f in os.listdir('/content/uncompressed/dev/'))
df = df[df['filename'].isin(available)].reset_index(drop=True)

print(df['label'].value_counts())
print(f"\nRatio: {df['label'].value_counts()['spoof'] / len(df) * 100:.1f}% spoof")

label
bonafide    2548
spoof        452
Name: count, dtype: int64

Ratio: 15.1% spoof


In [50]:
# Find a bonafide sample
proto_path = Path(ASV_ROOT) / 'LA' / 'ASVspoof2019_LA_cm_protocols' / 'ASVspoof2019.LA.cm.dev.trl.txt'
df = pd.read_csv(proto_path, sep=' ', header=None,
                 names=['speaker','filename','dash','system_id','label'])
available = set(f.replace('.pt','') for f in os.listdir('/content/uncompressed/dev/'))
df = df[df['filename'].isin(available)].reset_index(drop=True)

bonafide_file = df[df['label']=='bonafide'].iloc[0]['filename']
spoof_file    = df[df['label']=='spoof'].iloc[0]['filename']

for name, fname, model_name, m in [
    ('bonafide', bonafide_file, 'baseline', baseline),
    ('spoof',    spoof_file,    'baseline', baseline),
    ('bonafide', bonafide_file, 'grl_only', grl_only),
    ('spoof',    spoof_file,    'grl_only', grl_only),
]:
    pt = torch.load(f'/content/uncompressed/dev/{fname}.pt', weights_only=True).unsqueeze(0).to(device)
    with torch.no_grad():
        out = m(pt)
        prob = torch.softmax(out['deepfake_logits'], -1)
        print(f"{model_name} | {name:8s} | bonafide={prob[0,0]:.3f} spoof={prob[0,1]:.3f}")

baseline | bonafide | bonafide=0.706 spoof=0.294
baseline | spoof    | bonafide=0.008 spoof=0.992
grl_only | bonafide | bonafide=1.000 spoof=0.000
grl_only | spoof    | bonafide=0.001 spoof=0.999


In [52]:
import torch

# Load same filename from different codec folders and check if they're different
fname = 'LA_D_1006568'

uncompressed = torch.load(f'/content/uncompressed/dev/{fname}.pt', weights_only=True)
mp3_32 = torch.load(f'/content/mp3_32/dev/{fname}.pt', weights_only=True)

print("Are they identical?", torch.allclose(uncompressed, mp3_32))
print("Uncompressed mean:", uncompressed.mean().item())
print("MP3_32 mean:", mp3_32.mean().item())
print("Max difference:", (uncompressed - mp3_32).abs().max().item())

Are they identical? True
Uncompressed mean: -4.713249290944077e-05
MP3_32 mean: -4.713249290944077e-05
Max difference: 0.0


In [53]:
opus_16 = torch.load(f'/content/opus_16/dev/{fname}.pt', weights_only=True)
print("Opus_16 identical to uncompressed?", torch.allclose(uncompressed, opus_16))

Opus_16 identical to uncompressed? True


In [54]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from data_utils import apply_codec, CODEC_CONFIG, codec_tag
from evaluate import compute_eer

@torch.no_grad()
def eval_per_codec_live(model, asv_root, processed_root, device, split='dev', max_samples=1000):
    import os

    # Load protocol
    proto = {'dev': 'ASVspoof2019.LA.cm.dev.trl.txt'}
    proto_path = Path(asv_root) / 'LA' / 'ASVspoof2019_LA_cm_protocols' / proto[split]
    df = pd.read_csv(proto_path, sep=' ', header=None,
                     names=['speaker','filename','dash','system_id','label'])

    # Filter to available files
    available = set(f.replace('.pt','') for f in os.listdir(f'{processed_root}/uncompressed/{split}/'))
    df = df[df['filename'].isin(available)].reset_index(drop=True)
    if max_samples:
        df = df.sample(min(max_samples, len(df)), random_state=42).reset_index(drop=True)

    results = []
    for codec, bitrates in CODEC_CONFIG.items():
        for bitrate in bitrates:
            tag = codec_tag(codec, bitrate)
            all_scores, all_labels = [], []

            for i in range(0, len(df), 32):
                batch = df.iloc[i:i+32]
                wfs, labs = [], []
                for _, row in batch.iterrows():
                    pt = Path(processed_root) / 'uncompressed' / split / f"{row['filename']}.pt"
                    if not pt.exists():
                        continue
                    wav = torch.load(pt, weights_only=True).unsqueeze(0)  # (1, T)
                    wav = apply_codec(wav, 16000, codec, bitrate)          # apply compression
                    wfs.append(wav.squeeze(0))
                    labs.append(0 if row['label'] == 'bonafide' else 1)

                if not wfs:
                    continue
                wf_batch = torch.stack(wfs).to(device)
                s = torch.softmax(model(wf_batch)['deepfake_logits'], -1)[:, 1]
                all_scores.extend(s.cpu().numpy())
                all_labels.extend(labs)

            eer = compute_eer(np.array(all_labels), np.array(all_scores))
            results.append({'codec': tag, 'EER%': round(eer*100, 1)})
            print(f"  {tag:<15} EER={eer*100:.1f}%")

    return pd.DataFrame(results)

print("=== Baseline ===")
df_baseline = eval_per_codec_live(baseline, ASV_ROOT, PROCESSED_ROOT, device)

print("\n=== GRL Only ===")
df_grl = eval_per_codec_live(grl_only, ASV_ROOT, PROCESSED_ROOT, device)

=== Baseline ===
  uncompressed    EER=13.3%
  mp3_32          EER=13.0%
  mp3_64          EER=11.5%
  mp3_128         EER=11.1%
  aac_32          EER=13.3%
  aac_64          EER=13.0%
  aac_128         EER=12.4%
  opus_16         EER=90.7%
  opus_32         EER=57.0%
  opus_64         EER=53.2%

=== GRL Only ===
  uncompressed    EER=0.1%
  mp3_32          EER=0.7%
  mp3_64          EER=0.5%
  mp3_128         EER=0.1%


KeyboardInterrupt: 

In [56]:
import pandas as pd
from pathlib import Path
import os

proto_path = Path(ASV_ROOT) / 'LA' / 'ASVspoof2019_LA_cm_protocols' / 'ASVspoof2019.LA.cm.dev.trl.txt'
df = pd.read_csv(proto_path, sep=' ', header=None,
                 names=['speaker','filename','dash','system_id','label'])

available = set(f.replace('.pt','') for f in os.listdir(f'{PROCESSED_ROOT}/uncompressed/dev/'))
df = df[df['filename'].isin(available)].reset_index(drop=True)
df_eval = df.sample(min(1000, len(df)), random_state=42).reset_index(drop=True)

print(f"Eval samples: {len(df_eval)}")

Eval samples: 1000


In [57]:
# Quick check - just opus codecs for GRL
for bitrate in [16, 32, 64]:
    tag = f'opus_{bitrate}'
    all_scores, all_labels = [], []

    for i in range(0, len(df_eval), 32):
        batch = df_eval.iloc[i:i+32]
        wfs, labs = [], []
        for _, row in batch.iterrows():
            pt = Path(PROCESSED_ROOT) / 'uncompressed' / 'dev' / f"{row['filename']}.pt"
            if not pt.exists():
                continue
            wav = torch.load(pt, weights_only=True).unsqueeze(0)
            wav = apply_codec(wav, 16000, 'opus', bitrate)
            wfs.append(wav.squeeze(0))
            labs.append(0 if row['label'] == 'bonafide' else 1)

        if not wfs:
            continue
        wf_batch = torch.stack(wfs).to(device)
        with torch.no_grad():
            s = torch.softmax(grl_only(wf_batch)['deepfake_logits'], -1)[:, 1]
        all_scores.extend(s.cpu().numpy())
        all_labels.extend(labs)

    eer = compute_eer(np.array(all_labels), np.array(all_scores))
    print(f"GRL {tag}: EER={eer*100:.1f}%")

GRL opus_16: EER=97.4%
GRL opus_32: EER=81.7%
GRL opus_64: EER=70.0%


In [59]:
import torch
torch.cuda.empty_cache()

# Reduce batch size for contrastive training
BATCH_SIZE = 16  # down from 32

In [61]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

BATCH_SIZE = 8  # down to 8
MAX_LEN_SEC = 1.5  # shorter clips = less memory

In [63]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

BATCH_SIZE      = 4
EMBED_DIM       = 128   # down from 256
MAX_SAMPLES     = 1000  # down from 3000

In [65]:
import torch, gc

# Delete all loaded models
del baseline, full, grl_only
del model

gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")
print(f"Free memory: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

GPU memory cleared.
Free memory: 2.06 GB


In [67]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch, gc
gc.collect()
torch.cuda.empty_cache()

BATCH_SIZE      = 8
EMBED_DIM       = 256
MAX_SAMPLES     = 500   # drastically reduce

In [68]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import time
from pathlib import Path

from model import CodecRobustDetector, ContrastiveLoss
from data_utils import get_dataloaders, N_CODEC_CLASSES
from evaluate import compute_eer, compute_auc

# ── Config ────────────────────────────────────────────────
ALPHA           = 0.0    # GRL off
BETA            = 0.1    # Contrastive on
USE_CONTRASTIVE = True
EPOCHS          = 30
BATCH_SIZE      = 32
LR              = 3e-4
MAX_SAMPLES     = 3000
EMBED_DIM       = 256
SAVE_EVERY      = 1
RESULTS_DIR     = "/content/drive/MyDrive/CodecRobust/results/contrastive_only"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Data ──────────────────────────────────────────────────
loaders = get_dataloaders(
    asv_root=ASV_ROOT,
    preprocessed_root=PROCESSED_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=0,
    contrastive=USE_CONTRASTIVE,
    max_samples=MAX_SAMPLES,
    max_len_sec=1.5,
)

# ── Model ─────────────────────────────────────────────────
model = CodecRobustDetector(embed_dim=EMBED_DIM, n_codec_classes=N_CODEC_CLASSES).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR/20)
df_crit    = nn.CrossEntropyLoss()
codec_crit = nn.CrossEntropyLoss()
con_crit   = ContrastiveLoss(temperature=0.07)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

# ── Resume from latest checkpoint if exists ───────────────
out_dir     = Path(RESULTS_DIR)
start_epoch = 1
best_eer    = 1.0

latest_ckpt = sorted(out_dir.glob("checkpoint_epoch_*.pt"))
if latest_ckpt:
    ckpt = torch.load(latest_ckpt[-1], map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_eer    = ckpt['best_eer']
    print(f"Resumed from epoch {ckpt['epoch']}, best EER: {best_eer*100:.2f}%")
else:
    print("Starting fresh.")

# ── GRL lambda schedule ───────────────────────────────────
def grl_lambda(step, total_steps, lam_max=1.0):
    p = step / max(total_steps, 1)
    return lam_max * (2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0)

# ── Training loop ─────────────────────────────────────────
total_steps = EPOCHS * len(loaders["train"])
step = (start_epoch - 1) * len(loaders["train"])

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0 = time.time()
    for batch in loaders["train"]:
        lam = grl_lambda(step, total_steps)
        model.set_lambda(lam)
        wf        = batch["waveform"].to(device)
        labels    = batch["label"].to(device)
        codec_idx = batch["codec_idx"].to(device)
        out       = model(wf)
        loss      = df_crit(out["deepfake_logits"], labels) + ALPHA * codec_crit(out["codec_logits"], codec_idx)

        if USE_CONTRASTIVE and "waveform2" in batch:
            wf2   = batch["waveform2"].to(device)
            out2  = model(wf2)
            B     = wf.size(0)
            pairs = torch.stack([out["embedding"], out2["embedding"]], dim=1).view(2*B, -1)
            loss  = loss + BETA * con_crit(pairs)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        step += 1
    scheduler.step()

    # ── Dev eval ──────────────────────────────────────────
    model.eval()
    scores, labs = [], []
    with torch.no_grad():
        for batch in loaders["dev"]:
            s = torch.softmax(model(batch["waveform"].to(device))["deepfake_logits"], -1)[:, 1]
            scores.append(s.cpu())
            labs.append(batch["label"])
    scores = torch.cat(scores).numpy()
    labs   = torch.cat(labs).numpy()
    eer    = compute_eer(labs, scores)
    auc    = compute_auc(labs, scores)

    print(f"Epoch {epoch:02d}/{EPOCHS} | EER={eer*100:.2f}% AUC={auc:.4f} | time={time.time()-t0:.1f}s")

    if eer < best_eer:
        best_eer = eer
        torch.save(model.state_dict(), out_dir / "best_model.pt")
        print(f"  ✓ Best EER: {best_eer*100:.2f}% — saved")

    if epoch % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'eer': eer,
            'best_eer': best_eer,
        }, out_dir / f"checkpoint_epoch_{epoch:02d}.pt")
        print(f"  ✓ Checkpoint saved: epoch {epoch}")

print(f"\nDone. Best EER: {best_eer*100:.2f}%")

Device: cuda
  [DataLoader] Using preprocessed data from: /content
  [Filter] train: 25380 → 3000 (matched preprocessed files)
  [DataLoader] train: 3,000 samples | 10 codec variants available
  [Filter] dev: 24844 → 3000 (matched preprocessed files)
  [DataLoader] dev  : 3,000 samples | 10 codec variants available
  [DataLoader] WARN: could not load eval: No preprocessed folders found under /content. Run: python src/preprocess_codecs.py --asv_root <root>
Params: 550,948
Starting fresh.


OutOfMemoryError: CUDA out of memory. Tried to allocate 500.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 139.81 MiB is free. Including non-PyTorch memory, this process has 14.42 GiB memory in use. Of the allocated memory 14.07 GiB is allocated by PyTorch, and 217.98 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)